# Executive Decision Report - Analysis Notebook
Reproduces every figure in the report from `clean_dataset.csv` (11 order records, 3-18 Jan 2026).

**Sections:** 1 Audit, 2 Clean, 3 Derived metrics, 4 Revenue-to-cash bridge, 5 Drivers by product / segment / city, 6 Discounts, 7 Lever scenarios, 8 ROI model (assumptions).

In [1]:
import pandas as pd, numpy as np
from IPython.display import display
pd.set_option('display.width',200)
df = pd.read_csv('clean_dataset.csv')
print(df.shape)
df

(11, 14)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,order_date_year,order_date_month,order_date_month_name,order_date_quarter,calculated_sales
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2.0,799,10.0,Paid,2026.0,1.0,January,1.0,1598.0
1,RT-1002,NaN,Fresher,Bengaluru,Course Access,1.0,1499,0.0,paid,NaN,NaN,NaN,NaN,1499.0
2,RT-1003,2026-01-05,student,Chennai,Course Access,1.0,1499,2.5,Pending,2026.0,1.0,January,1.0,1499.0
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3.0,799,5.0,Paid,2026.0,1.0,January,1.0,2397.0
4,RT-1005,2026-01-09,Fresher,Chennai,Mentor Session,1.0,999,0.0,Failed,2026.0,1.0,January,1.0,999.0
5,RT-1006,NaN,Student,Pune,Learning Kit,-0.5,799,10.0,Paid,NaN,NaN,NaN,NaN,-399.5
6,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2.0,999,25.0,Paid,2026.0,1.0,January,1.0,1998.0
7,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,1.0,1499,0.0,Pending,2026.0,1.0,January,1.0,1499.0
8,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1.0,799,0.0,Paid,2026.0,1.0,January,1.0,799.0
9,RT-1010,2026-01-18,Professional,Delhi,Course Access,2.0,1499,15.0,Refunded,2026.0,1.0,January,1.0,2998.0


## 1. Audit: what is wrong with the file

In [2]:
print('Missing values:\n', df.isna().sum()[lambda s: s>0])
print('\nNegative quantity rows:', df.loc[df.quantity<=0,'order_id'].tolist())
print('Segment labels :', df.customer_segment.unique())
print('Status labels  :', df.payment_status.unique())
print('Rows where calculated_sales ignores a discount:', int((df.discount_pct>0).sum()))

Missing values:
 order_date               3
order_date_year          3
order_date_month         3
order_date_month_name    3
order_date_quarter       3
dtype: int64

Negative quantity rows: ['RT-1006']
Segment labels : <StringArray>
['Student', 'Fresher', 'student', 'Professional']
Length: 4, dtype: str
Status labels  : <StringArray>
['Paid', 'paid', 'Pending', 'Failed', 'Refunded']
Length: 5, dtype: str
Rows where calculated_sales ignores a discount: 6


## 2. Clean
Standardise label casing; flag the invalid (negative-quantity) order and exclude it from financial metrics.

In [3]:
df['customer_segment'] = df.customer_segment.str.strip().str.title()
df['payment_status'] = df.payment_status.str.strip().str.title()
df['order_date'] = pd.to_datetime(df.order_date)
df['valid'] = df.quantity > 0
print(len(df), 'rows ->', int(df.valid.sum()), 'valid orders')

11 rows -> 10 valid orders


## 3. Derived metrics
Gross = quantity x unit price. Discount = gross x discount %. Net billed = gross - discount. Collected = net on Paid orders.

In [4]:
df['gross'] = df.quantity*df.unit_price
df['discount_amt'] = df.gross*df.discount_pct/100
df['net_billed'] = df.gross-df.discount_amt
df['collected'] = np.where(df.payment_status.eq('Paid'), df.net_billed, 0.0)
v = df[df.valid].copy()
M = dict(gross=v.gross.sum(), discount=v.discount_amt.sum(), net=v.net_billed.sum(), collected=v.collected.sum())
M['discount_pct']=M['discount']/M['gross']*100; M['collection_rate']=M['collected']/M['net']*100
print({k: round(x,1) for k,x in M.items()})
print('Raw calculated_sales total (incl. invalid row):', df.calculated_sales.sum())

{'gross': np.float64(16285.0), 'discount': np.float64(1266.3), 'net': np.float64(15018.7), 'collected': np.float64(8510.8), 'discount_pct': np.float64(7.8), 'collection_rate': np.float64(56.7)}
Raw calculated_sales total (incl. invalid row): 15885.5


## 4. Revenue-to-cash bridge

In [5]:
bridge = v.groupby('payment_status').net_billed.sum()
print(bridge.round(1))
assert abs(bridge.sum()-M['net'])<0.01, 'bridge must reconcile to net billed'
print('Reconciles. At risk (pending+failed+refunded):', round(M['net']-M['collected'],1), f"({(M['net']-M['collected'])/M['net']*100:.1f}%)")

payment_status
Failed       999.0
Paid        8510.8
Pending     2960.5
Refunded    2548.3
Name: net_billed, dtype: float64
Reconciles. At risk (pending+failed+refunded): 6507.8 (43.3%)


## 5. Drivers by product, segment, city

In [6]:
def by(col):
    g = v.groupby(col).agg(orders=('order_id','count'), units=('quantity','sum'), gross=('gross','sum'), discount=('discount_amt','sum'), net=('net_billed','sum'), collected=('collected','sum'))
    g['discount_pct']=g.discount/g.gross*100; g['collection_pct']=g.collected/g.net*100
    return g.round(1)
for c_ in ['category','customer_segment','city']:
    display(by(c_))

,orders,units,gross,discount,net,collected,discount_pct,collection_pct
category,,,,,,,,
Course Access,4,5.0,7495.0,487.2,7007.8,1499.0,6.5,21.4
Learning Kit,3,6.0,4794.0,279.6,4514.4,4514.4,5.8,100.0
Mentor Session,3,4.0,3996.0,499.5,3496.5,2497.5,12.5,71.4


,orders,units,gross,discount,net,collected,discount_pct,collection_pct
customer_segment,,,,,,,,
Fresher,3,3.0,3997.0,0.0,3997.0,1499.0,0.0,37.5
Professional,3,7.0,7393.0,1069.0,6324.0,3775.6,14.5,59.7
Student,4,5.0,4895.0,197.3,4697.7,3236.2,4.0,68.9


,orders,units,gross,discount,net,collected,discount_pct,collection_pct
city,,,,,,,,
Bengaluru,2,2.0,2998.0,0.0,2998.0,1499.0,0.0,50.0
Chennai,4,5.0,4895.0,197.3,4697.7,2237.2,4.0,47.6
Delhi,1,2.0,2998.0,449.7,2548.3,0.0,15.0,0.0
Hyderabad,1,3.0,2397.0,119.8,2277.2,2277.2,5.0,100.0
Kochi,1,1.0,999.0,0.0,999.0,999.0,0.0,100.0
Mumbai,1,2.0,1998.0,499.5,1498.5,1498.5,25.0,100.0


## 6. Discount concentration

In [7]:
top2 = v.nlargest(2,'discount_amt')[['order_id','city','category','discount_pct','discount_amt','payment_status']]
display(top2)
print('Top-2 share of all discount: %.0f%%' % (top2.discount_amt.sum()/M['discount']*100))
disc = v[v.discount_pct>0]; full = v[v.discount_pct==0]
print('Collection rate, discounted orders: %.1f%%' % (disc.collected.sum()/disc.net_billed.sum()*100))
print('Collection rate, full-price orders: %.1f%%' % (full.collected.sum()/full.net_billed.sum()*100))
v['cap_save'] = np.where(v.discount_pct>10,(v.discount_pct-10)/100*v.gross,0)
print('Saved by a 10%% cap: all=%.1f, on paid orders=%.1f' % (v.cap_save.sum(), v.loc[v.payment_status.eq('Paid'),'cap_save'].sum()))

,order_id,city,category,discount_pct,discount_amt,payment_status
6,RT-1007,Mumbai,Mentor Session,25.0,499.5,Paid
9,RT-1010,Delhi,Course Access,15.0,449.7,Refunded


Top-2 share of all discount: 75%
Collection rate, discounted orders: 56.5%
Collection rate, full-price orders: 56.9%
Saved by a 10% cap: all=449.6, on paid orders=299.7


## 7. Lever scenarios (sample values)

In [8]:
pend=bridge['Pending']; fail=bridge['Failed']; ref=bridge['Refunded']; cap_paid=v.loc[v.payment_status.eq('Paid'),'cap_save'].sum()
levers = {'Conservative':dict(p=.5,f=0,r=0,d=.5),'Base':dict(p=.75,f=.25,r=.25,d=.75),'Stretch':dict(p=.9,f=.5,r=.5,d=1.0)}
S={}
for k,l in levers.items():
    S[k]=dict(rec1=l['p']*pend+l['f']*fail, rec2=l['d']*cap_paid, rec3=l['r']*ref)
    S[k]['total']=sum(S[k].values())
pd.DataFrame(S).T.round(1)

,rec1,rec2,rec3,total
Conservative,1480.3,149.8,0.0,1630.1
Base,2470.1,224.8,637.1,3332.0
Stretch,3164.0,299.7,1274.2,4737.8


## 8. ROI model
**Assumptions, not source data - replace with real quotes.** Sample window = 16 days (3-18 Jan) annualised by 365/16; year-1 ramp 75%; costs below are placeholders; undiscounted.

In [9]:
F = 365/16; RAMP=[.75,1,1]
COST = {'rec1':dict(one=20000,run=12000),'rec2':dict(one=5000,run=0),'rec3':dict(one=12000,run=3000)}
def payback(one,run,ann):
    cum=-one
    for m in range(1,37):
        cum += ann/12*RAMP[(m-1)//12] - run/12
        if cum>=0: return m
    return None
rows=[]
for k in S:
    tot=dict(cost=0,ben=0,one=0,run=0,ann=0)
    for r,c_ in COST.items():
        ann=S[k][r]*F; ben=ann*sum(RAMP); cost=c_['one']+3*c_['run']
        rows.append((k,r,round(cost),round(ben),round(ben-cost),round((ben-cost)/cost*100),payback(c_['one'],c_['run'],ann)))
        tot['cost']+=cost; tot['ben']+=ben; tot['one']+=c_['one']; tot['run']+=c_['run']; tot['ann']+=ann
    rows.append((k,'TOTAL',round(tot['cost']),round(tot['ben']),round(tot['ben']-tot['cost']),round((tot['ben']-tot['cost'])/tot['cost']*100),payback(tot['one'],tot['run'],tot['ann'])))
pd.DataFrame(rows,columns=['scenario','item','cost_3yr','benefit_3yr','net','roi_pct','payback_months'])

,scenario,item,cost_3yr,benefit_3yr,net,roi_pct,payback_months
0,Conservative,rec1,56000,92863,36863,66,16.0
1,Conservative,rec2,5000,9401,4401,88,21.0
2,Conservative,rec3,21000,0,-21000,-100,NaN
3,Conservative,TOTAL,82000,102264,20264,25,26.0
4,Base,rec1,56000,154963,98963,177,8.0
5,Base,rec2,5000,14101,9101,182,15.0
6,Base,rec3,21000,39967,18967,90,17.0
7,Base,TOTAL,82000,209031,127031,155,11.0
8,Stretch,rec1,56000,198490,142490,254,6.0
9,Stretch,rec2,5000,18801,13801,276,12.0


In [10]:
# Export the cleaned order-level table used in the report
out = df[['order_id','order_date','customer_segment','city','category','quantity','unit_price','discount_pct','payment_status','valid','gross','discount_amt','net_billed','collected']].round(2)
out.to_csv('clean_orders.csv', index=False)
out

/tmp/ipykernel_165/1342932700.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  out = df[['order_id','order_date','customer_segment','city','category','quantity','unit_price','discount_pct','payment_status','valid','gross','discount_amt','net_billed','collected']].round(2)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,valid,gross,discount_amt,net_billed,collected
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2.0,799,10.0,Paid,True,1598.0,159.80,1438.20,1438.20
1,RT-1002,NaT,Fresher,Bengaluru,Course Access,1.0,1499,0.0,Paid,True,1499.0,0.00,1499.00,1499.00
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1.0,1499,2.5,Pending,True,1499.0,37.48,1461.52,0.00
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3.0,799,5.0,Paid,True,2397.0,119.85,2277.15,2277.15
4,RT-1005,2026-01-09,Fresher,Chennai,Mentor Session,1.0,999,0.0,Failed,True,999.0,0.00,999.00,0.00
5,RT-1006,NaT,Student,Pune,Learning Kit,-0.5,799,10.0,Paid,False,-399.5,-39.95,-359.55,-359.55
6,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2.0,999,25.0,Paid,True,1998.0,499.50,1498.50,1498.50
7,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,1.0,1499,0.0,Pending,True,1499.0,0.00,1499.00,0.00
8,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1.0,799,0.0,Paid,True,799.0,0.00,799.00,799.00
9,RT-1010,2026-01-18,Professional,Delhi,Course Access,2.0,1499,15.0,Refunded,True,2998.0,449.70,2548.30,0.00
